# 07 Coarse grid data (256³)

Builds 256³ versions of the TNG300-3 dark matter and subhalo fields (2×2×2 block means) and a 256³ T-web mask recomputed from the dark matter with σ_eff = 1 h⁻¹ Mpc. Then it checks how much the coarser grid costs on its own: the 256³ mask, upsampled to 512³, is scored against the 512³ mask. That is the best any 256³ model could score at 512³.

Run once. Afterwards train `A256_scratch_L10` in notebook 05.

Runtime: CPU is enough, with **High-RAM**. A few minutes.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get('GITHUB_TOKEN')
!rm -rf /content/deepvoid
!git clone -q "https://{token}@github.com/skumagai3/deepvoid.git" /content/deepvoid
!git -C /content/deepvoid remote set-url origin https://github.com/skumagai3/deepvoid.git
!pip -q install --force-reinstall --no-deps /content/deepvoid
del token
!git -C /content/deepvoid log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Write the 256³ files

In [ ]:
!python /content/deepvoid/scripts/make_coarse.py

## 2. Ceiling: a perfect 256³ prediction scored at 512³

On the test slab only, as in training runs.

In [ ]:
import numpy as np
from deepvoid import Catalog
from deepvoid.evaluate import confusion_matrix, scores
from deepvoid.resample import upsampled_argmax
from deepvoid.split import slab_split

catalog = Catalog()
fine = catalog.load('tng300-3/mask').data
coarse = catalog.load('tng300-3/n256/mask').data
onehot = np.eye(4, dtype=np.float32)[coarse]
up = upsampled_argmax(onehot, 2)
test = slab_split(512, buffer=16)['test'].mask(512)
s = scores(confusion_matrix(fine, up, mask=test))
print({k: round(v, 3) for k, v in s.items() if k in ('mcc', 'void_f1', 'wall_f1', 'filament_f1', 'halo_f1')})
print('class fractions 512:', np.round(np.bincount(fine.ravel(), minlength=4) / fine.size, 3))
print('class fractions 256:', np.round(np.bincount(coarse.ravel(), minlength=4) / coarse.size, 3))

Paste the output back to Claude. For comparison, A scored MCC 0.632 at λ = 10, so a ceiling far above that means the coarse grid itself is not the limit.